In [2]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Variables
distance = ctrl.Antecedent(np.arange(0, 51, 1), 'distance')
traffic = ctrl.Antecedent(np.arange(0, 101, 1), 'traffic')
demand = ctrl.Antecedent(np.arange(0, 101, 1), 'demand')
weather = ctrl.Antecedent(np.arange(0, 3, 1), 'weather')
price = ctrl.Consequent(np.arange(0, 4, 1), 'price')
reward = ctrl.Consequent(np.arange(0, 4, 1), 'reward')

# Membership functions
distance['short'] = fuzz.trimf(distance.universe, [0, 0, 3])
distance['medium'] = fuzz.trimf(distance.universe, [2, 5, 8])
distance['long'] = fuzz.trimf(distance.universe, [6, 13, 20])
distance['very_long'] = fuzz.trimf(distance.universe, [15, 32, 50])

traffic['low'] = fuzz.trimf(traffic.universe, [0, 0, 30])
traffic['medium'] = fuzz.trimf(traffic.universe, [20, 45, 70])
traffic['high'] = fuzz.trimf(traffic.universe, [60, 100, 100])

demand['low'] = fuzz.trimf(demand.universe, [0, 0, 30])
demand['medium'] = fuzz.trimf(demand.universe, [20, 45, 70])
demand['high'] = fuzz.trimf(demand.universe, [60, 100, 100])

weather['good'] = fuzz.trimf(weather.universe, [0, 0, 1])
weather['moderate'] = fuzz.trimf(weather.universe, [0, 1, 2])
weather['bad'] = fuzz.trimf(weather.universe, [1, 2, 2])

price['low'] = fuzz.trimf(price.universe, [0, 0, 1])
price['medium'] = fuzz.trimf(price.universe, [0, 1, 2])
price['high'] = fuzz.trimf(price.universe, [1, 2, 3])
price['very_high'] = fuzz.trimf(price.universe, [2, 3, 3])

reward['none'] = fuzz.trimf(reward.universe, [0, 0, 0.5])
reward['few'] = fuzz.trimf(reward.universe, [0, 1, 2])
reward['moderate'] = fuzz.trimf(reward.universe, [1, 2, 3])
reward['high'] = fuzz.trimf(reward.universe, [2, 3, 3])

# Rules
rules = [
    ctrl.Rule(distance['short'] & traffic['low'] & demand['low'], price['low']),
    ctrl.Rule(distance['short'] & traffic['medium'] & demand['high'], price['medium']),
    ctrl.Rule(distance['medium'] & traffic['high'] & demand['high'], price['high']),
    ctrl.Rule(distance['long'] & traffic['medium'] & weather['good'], price['medium']),
    ctrl.Rule(distance['long'] & traffic['high'] & weather['bad'], price['very_high']),
    ctrl.Rule(distance['very_long'] & traffic['high'] & demand['high'], price['very_high']),
    ctrl.Rule(distance['medium'] & traffic['low'] & demand['low'], price['medium']),
    ctrl.Rule(distance['short'] & traffic['high'] & weather['bad'], price['high']),
    ctrl.Rule(distance['very_long'] & weather['bad'], price['very_high']),
    ctrl.Rule(distance['medium'] & traffic['medium'] & weather['moderate'], price['medium']),
]

ctrl_sys = ctrl.ControlSystem(rules)
simulation = ctrl.ControlSystemSimulation(ctrl_sys)

def calculate(distance_val, traffic_val, demand_val, weather_val):
    simulation.input['distance'] = distance_val
    simulation.input['traffic'] = traffic_val
    simulation.input['demand'] = demand_val
    simulation.input['weather'] = weather_val
    simulation.compute()
    
    price_labels = ['Thấp', 'Trung bình', 'Cao', 'Rất cao']
    return simulation.output['price'], price_labels[int(round(simulation.output['price']))]

# Test
if __name__ == "__main__":
    print("=== GRAB-BIKE PRICING ===")
    
    for i, (d, t, dm, w) in enumerate([
        (2, 20, 25, 0),    # Short, low traffic, low demand
        (15, 80, 75, 2),   # Long, high traffic, bad weather
        (6, 50, 50, 1),    # Medium
    ], 1):
        price_val, price_cat = calculate(d, t, dm, w)
        print(f"\nTest {i}: Distance={d}km, Traffic={t}%, Demand={dm}%, Weather={w}")
        print(f"→ Price: {price_val:.2f} ({price_cat})")

=== GRAB-BIKE PRICING ===

Test 1: Distance=2km, Traffic=20%, Demand=25%, Weather=0
→ Price: 0.46 (Thấp)

Test 2: Distance=15km, Traffic=80%, Demand=75%, Weather=2
→ Price: 2.61 (Rất cao)

Test 3: Distance=6km, Traffic=50%, Demand=50%, Weather=1
→ Price: 1.00 (Trung bình)


In [3]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Input variables
store_rating = ctrl.Antecedent(np.arange(0, 6, 0.1), 'store_rating')  # 1-5 sao
sales_volume = ctrl.Antecedent(np.arange(0, 101, 1), 'sales_volume')  # %
profit_margin = ctrl.Antecedent(np.arange(0, 101, 1), 'profit_margin')  # %
seasonal_event = ctrl.Antecedent(np.arange(0, 4, 1), 'seasonal_event')  # 0-3
competitor_discount = ctrl.Antecedent(np.arange(0, 4, 1), 'competitor_discount')  # 0-3

# Output variable
discount_rate = ctrl.Consequent(np.arange(0, 71, 1), 'discount_rate')  # 0-70%

# Membership functions - Inputs
store_rating['low'] = fuzz.trimf(store_rating.universe, [1.0, 1.0, 4.0])
store_rating['medium'] = fuzz.trimf(store_rating.universe, [3.5, 4.25, 5.0])
store_rating['high'] = fuzz.trimf(store_rating.universe, [4.5, 5.0, 5.0])

sales_volume['low'] = fuzz.trimf(sales_volume.universe, [0, 0, 30])
sales_volume['medium'] = fuzz.trimf(sales_volume.universe, [20, 50, 80])
sales_volume['high'] = fuzz.trimf(sales_volume.universe, [60, 100, 100])

profit_margin['low'] = fuzz.trimf(profit_margin.universe, [0, 0, 30])
profit_margin['medium'] = fuzz.trimf(profit_margin.universe, [20, 50, 80])
profit_margin['high'] = fuzz.trimf(profit_margin.universe, [60, 100, 100])

seasonal_event['none'] = fuzz.trimf(seasonal_event.universe, [0, 0, 1])
seasonal_event['moderate'] = fuzz.trimf(seasonal_event.universe, [0, 1, 2])
seasonal_event['high'] = fuzz.trimf(seasonal_event.universe, [1, 2, 3])

competitor_discount['low'] = fuzz.trimf(competitor_discount.universe, [0, 0, 1])
competitor_discount['medium'] = fuzz.trimf(competitor_discount.universe, [0, 1, 2])
competitor_discount['high'] = fuzz.trimf(competitor_discount.universe, [1, 2, 3])

# Membership functions - Output
discount_rate['very_low'] = fuzz.trimf(discount_rate.universe, [0, 0, 5])
discount_rate['low'] = fuzz.trimf(discount_rate.universe, [0, 5, 10])
discount_rate['medium'] = fuzz.trimf(discount_rate.universe, [10, 15, 20])
discount_rate['high'] = fuzz.trimf(discount_rate.universe, [20, 30, 40])
discount_rate['very_high'] = fuzz.trimf(discount_rate.universe, [40, 55, 70])

# Fuzzy Rules from the book
rules = [
    # Rule 1: High rating, high sales, high profit → Very low discount
    ctrl.Rule(store_rating['high'] & sales_volume['high'] & profit_margin['high'], 
              discount_rate['very_low']),
    
    # Rule 2: Low rating, low sales, high profit → High discount
    ctrl.Rule(store_rating['low'] & sales_volume['low'] & profit_margin['high'], 
              discount_rate['high']),
    
    # Rule 3: High seasonal event, high competitor discount → Very high discount
    ctrl.Rule(seasonal_event['high'] & competitor_discount['high'], 
              discount_rate['very_high']),
    
    # Rule 4: Medium rating, medium sales, medium profit → Medium discount
    ctrl.Rule(store_rating['medium'] & sales_volume['medium'] & profit_margin['medium'], 
              discount_rate['medium']),
    
    # Rule 5: Low competitor discount, low profit, high sales → Very low discount
    ctrl.Rule(competitor_discount['low'] & profit_margin['low'] & sales_volume['high'], 
              discount_rate['very_low']),
    
    # Rule 6: Low rating, no seasonal event → Medium discount
    ctrl.Rule(store_rating['low'] & seasonal_event['none'], 
              discount_rate['medium']),
    
    # Rule 7: Low sales, low profit → Very high discount
    ctrl.Rule(sales_volume['low'] & profit_margin['low'], 
              discount_rate['very_high']),
]

# Create control system
ctrl_sys = ctrl.ControlSystem(rules)
simulation = ctrl.ControlSystemSimulation(ctrl_sys)

def calculate_discount(rating, sales_vol, profit, season_level, competitor_level):
    """
    Calculate optimal discount rate for Shopee store
    
    Parameters:
    - rating: Store rating (1.0-5.0 stars)
    - sales_vol: Sales volume (0-100%)
    - profit: Profit margin (0-100%)
    - season_level: Seasonal event (0=None, 1=Moderate, 2-3=High)
    - competitor_level: Competitor discount (0=Low, 1=Moderate, 2-3=High)
    
    Returns:
    - discount_value: Calculated discount percentage
    - discount_category: Category label
    """
    simulation.input['store_rating'] = rating
    simulation.input['sales_volume'] = sales_vol
    simulation.input['profit_margin'] = profit
    simulation.input['seasonal_event'] = season_level
    simulation.input['competitor_discount'] = competitor_level
    
    simulation.compute()
    
    discount_value = simulation.output['discount_rate']
    
    # Categorize
    if discount_value <= 5:
        category = "Rất thấp (0-5%)"
    elif discount_value <= 10:
        category = "Thấp (5-10%)"
    elif discount_value <= 20:
        category = "Trung bình (10-20%)"
    elif discount_value <= 40:
        category = "Cao (20-40%)"
    else:
        category = "Rất cao (40-70%)"
    
    return discount_value, category

# Test cases
if __name__ == "__main__":
    print("=== SHOPEE DISCOUNT STRATEGY ===\n")
    
    test_cases = [
        # (rating, sales_vol, profit, season, competitor, description)
        (4.8, 85, 70, 0, 1, "Store tốt, doanh số cao, lợi nhuận cao"),
        (3.5, 25, 60, 0, 1, "Store mới, doanh số thấp, lợi nhuận cao"),
        (4.2, 50, 40, 3, 3, "Sự kiện lớn (11.11), đối thủ giảm giá mạnh"),
        (4.0, 50, 50, 1, 1, "Điều kiện trung bình"),
        (3.8, 30, 20, 0, 0, "Doanh số thấp, lợi nhuận thấp"),
    ]
    
    for i, (rating, sales, profit, season, competitor, desc) in enumerate(test_cases, 1):
        discount_val, discount_cat = calculate_discount(
            rating, sales, profit, season, competitor
        )
        print(f"Test {i}: {desc}")
        print(f"  Rating: {rating}, Sales: {sales}%, Profit: {profit}%")
        print(f"  Season: {season}, Competitor: {competitor}")
        print(f"  → Discount: {discount_val:.1f}% - {discount_cat}\n")

=== SHOPEE DISCOUNT STRATEGY ===

Test 1: Store tốt, doanh số cao, lợi nhuận cao
  Rating: 4.8, Sales: 85%, Profit: 70%
  Season: 0, Competitor: 1
  → Discount: 2.2% - Rất thấp (0-5%)

Test 2: Store mới, doanh số thấp, lợi nhuận cao
  Rating: 3.5, Sales: 25%, Profit: 60%
  Season: 0, Competitor: 1
  → Discount: 15.0% - Trung bình (10-20%)

Test 3: Sự kiện lớn (11.11), đối thủ giảm giá mạnh
  Rating: 4.2, Sales: 50%, Profit: 40%
  Season: 3, Competitor: 3
  → Discount: 15.0% - Trung bình (10-20%)

Test 4: Điều kiện trung bình
  Rating: 4.0, Sales: 50%, Profit: 50%
  Season: 1, Competitor: 1
  → Discount: 15.0% - Trung bình (10-20%)

Test 5: Doanh số thấp, lợi nhuận thấp
  Rating: 3.8, Sales: 30%, Profit: 20%
  Season: 0, Competitor: 0
  → Discount: 15.0% - Trung bình (10-20%)



In [13]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ==========================================
# 2.13: CHIẾN LƯỢC SHOPEE - MẶT HÀNG ĐẶC BIỆT
# ==========================================

# Inputs
demand = ctrl.Antecedent(np.arange(0, 101, 1), 'demand')
competitor = ctrl.Antecedent(np.arange(0, 101, 1), 'competitor')
reputation = ctrl.Antecedent(np.arange(0, 101, 1), 'reputation')
profit = ctrl.Antecedent(np.arange(0, 101, 1), 'profit')
season = ctrl.Antecedent(np.arange(0, 101, 1), 'season')

# Output
discount = ctrl.Consequent(np.arange(0, 71, 1), 'discount')

# Membership functions
demand['low'] = fuzz.trimf(demand.universe, [0, 0, 30])
demand['medium'] = fuzz.trimf(demand.universe, [20, 50, 80])
demand['high'] = fuzz.trimf(demand.universe, [60, 100, 100])

competitor['low'] = fuzz.trimf(competitor.universe, [0, 0, 30])
competitor['medium'] = fuzz.trimf(competitor.universe, [20, 50, 80])
competitor['high'] = fuzz.trimf(competitor.universe, [60, 100, 100])

reputation['low'] = fuzz.trimf(reputation.universe, [0, 0, 30])
reputation['medium'] = fuzz.trimf(reputation.universe, [20, 50, 80])
reputation['high'] = fuzz.trimf(reputation.universe, [60, 100, 100])

profit['low'] = fuzz.trimf(profit.universe, [0, 0, 30])
profit['medium'] = fuzz.trimf(profit.universe, [20, 50, 80])
profit['high'] = fuzz.trimf(profit.universe, [60, 100, 100])

season['low'] = fuzz.trimf(season.universe, [0, 0, 30])
season['medium'] = fuzz.trimf(season.universe, [20, 50, 80])
season['high'] = fuzz.trimf(season.universe, [60, 100, 100])

discount['very_low'] = fuzz.trimf(discount.universe, [0, 0, 5])
discount['low'] = fuzz.trimf(discount.universe, [0, 5, 10])
discount['medium'] = fuzz.trimf(discount.universe, [5, 15, 20])
discount['high'] = fuzz.trimf(discount.universe, [15, 30, 40])
discount['very_high'] = fuzz.trimf(discount.universe, [30, 55, 70])

# Rules (Dựa trên sách trang 91)
rules = [
    # 1. Nhu cầu cao, Áp lực thấp, Lợi nhuận thấp -> Rất thấp
    ctrl.Rule(demand['high'] & competitor['low'] & profit['low'], discount['very_low']),
    # 2. Nhu cầu thấp, Áp lực cao, Lợi nhuận cao -> Cao
    ctrl.Rule(demand['low'] & competitor['high'] & profit['high'], discount['high']),
    # 3. Uy tín cao, Lợi nhuận TB, Mùa cao -> Trung bình
    ctrl.Rule(reputation['high'] & profit['medium'] & season['high'], discount['medium']),
    # 4. Áp lực cao, Mùa cao, Lợi nhuận cao -> Rất cao
    ctrl.Rule(competitor['high'] & season['high'] & profit['high'], discount['very_high']),
    # 5. Uy tín thấp, Nhu cầu TB, Lợi nhuận thấp -> Trung bình
    ctrl.Rule(reputation['low'] & demand['medium'] & profit['low'], discount['medium']),
    # 6. Nhu cầu cao, Mùa không có, Áp lực thấp -> Rất thấp
    ctrl.Rule(demand['high'] & season['low'] & competitor['low'], discount['very_low']),
    # 7. Lợi nhuận cao, Áp lực TB, Mùa TB -> Trung bình
    ctrl.Rule(profit['high'] & competitor['medium'] & season['medium'], discount['medium'])
]

system = ctrl.ControlSystem(rules)
sim = ctrl.ControlSystemSimulation(system)

def calculate_discount(demand_val, comp_val, rep_val, profit_val, season_val):
    sim.input['demand'] = demand_val
    sim.input['competitor'] = comp_val
    sim.input['reputation'] = rep_val
    sim.input['profit'] = profit_val
    sim.input['season'] = season_val
    
    try:
        sim.compute()
        # Kiểm tra xem key 'discount' có tồn tại không (tránh lỗi nếu không luật nào khớp)
        if 'discount' in sim.output:
            return sim.output['discount']
        else:
            return None
    except Exception as e:
        print(f"Lỗi tính toán: {e}")
        return None

if __name__ == "__main__":
    print("=== 2.13 SHOPEE DISCOUNT STRATEGY ===")
    
    # Test 1: Kích hoạt Luật 4 (Áp lực cao, Mùa cao, Lợi nhuận cao -> Rất cao)
    # Đầu vào: Demand(50), Comp(80-High), Rep(50), Profit(80-High), Season(80-High)
    res1 = calculate_discount(50, 80, 50, 80, 80)
    print(f"Test 1 (Luật 4): Discount = {res1}% (Kì vọng: ~40-70%)\n")

    # Test 2: Kích hoạt Luật 1 (Nhu cầu cao, Áp lực thấp, Lợi nhuận thấp -> Rất thấp)
    # Đầu vào: Demand(80-High), Comp(20-Low), Rep(50), Profit(20-Low), Season(50)
    res2 = calculate_discount(80, 20, 50, 20, 50)
    print(f"Test 2 (Luật 1): Discount = {res2}% (Kì vọng: ~0-5%)\n")
    
    # Test 3: Kích hoạt Luật 7 (Lợi nhuận cao, Áp lực TB, Mùa TB -> Trung bình)
    # Đầu vào: Demand(50), Comp(50-Med), Rep(50), Profit(80-High), Season(50-Med)
    res3 = calculate_discount(50, 50, 50, 80, 50)
    print(f"Test 3 (Luật 7): Discount = {res3}% (Kì vọng: ~10-20%)")

=== 2.13 SHOPEE DISCOUNT STRATEGY ===
Test 1 (Luật 4): Discount = 51.1111111111111% (Kì vọng: ~40-70%)

Test 2 (Luật 1): Discount = 2.111111111111111% (Kì vọng: ~0-5%)

Test 3 (Luật 7): Discount = 13.055555555555557% (Kì vọng: ~10-20%)


In [14]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# ==========================================
# 2.14: TỐI ƯU HÓA GIAO HÀNG
# ==========================================

# Inputs
density = ctrl.Antecedent(np.arange(0, 101, 1), 'density')
urgency = ctrl.Antecedent(np.arange(0, 101, 1), 'urgency')
load = ctrl.Antecedent(np.arange(0, 101, 1), 'load')
traffic = ctrl.Antecedent(np.arange(0, 101, 1), 'traffic')
profit_del = ctrl.Antecedent(np.arange(0, 101, 1), 'profit_del')

# Outputs
combine = ctrl.Consequent(np.arange(0, 101, 1), 'combine')
priority = ctrl.Consequent(np.arange(0, 101, 1), 'priority')

# Membership functions
for var in [density, urgency, load, traffic, profit_del]:
    var['low'] = fuzz.trimf(var.universe, [0, 0, 30])
    var['medium'] = fuzz.trimf(var.universe, [20, 50, 80])
    var['high'] = fuzz.trimf(var.universe, [60, 100, 100])

combine['few'] = fuzz.trimf(combine.universe, [0, 0, 30])
combine['some'] = fuzz.trimf(combine.universe, [20, 50, 80])
combine['many'] = fuzz.trimf(combine.universe, [60, 100, 100])

priority['low'] = fuzz.trimf(priority.universe, [0, 0, 30])
priority['medium'] = fuzz.trimf(priority.universe, [20, 50, 80])
priority['high'] = fuzz.trimf(priority.universe, [60, 100, 100])

# Rules
rules = [
    # Luật kết hợp (1-5)
    ctrl.Rule(density['high'] & load['low'] & traffic['low'], combine['many']),
    ctrl.Rule(density['medium'] & traffic['high'] & urgency['medium'], combine['some']),
    ctrl.Rule(load['high'] & density['high'] & profit_del['medium'], combine['some']),
    ctrl.Rule(density['low'] & urgency['high'] & traffic['medium'], combine['some']),
    ctrl.Rule(profit_del['high'] & urgency['high'] & traffic['high'], combine['some']),
    
    # Luật ưu tiên (6-8)
    ctrl.Rule(urgency['high'] & profit_del['high'], priority['high']),
    ctrl.Rule(urgency['medium'] & traffic['medium'], priority['medium']),
    ctrl.Rule(urgency['low'] & density['high'] & profit_del['low'], priority['low'])
]

system = ctrl.ControlSystem(rules)
sim = ctrl.ControlSystemSimulation(system)

def optimize_delivery(den_val, urg_val, load_val, traf_val, prof_val):
    sim.input['density'] = den_val
    sim.input['urgency'] = urg_val
    sim.input['load'] = load_val
    sim.input['traffic'] = traf_val
    sim.input['profit_del'] = prof_val
    
    try:
        sim.compute()
        c_val = sim.output.get('combine', None)
        p_val = sim.output.get('priority', None)
        return c_val, p_val
    except Exception:
        return None, None

if __name__ == "__main__":
    print("=== 2.14 DELIVERY OPTIMIZATION ===")
    
    # Test Case: Kích hoạt Luật 1 (Kết hợp Nhiều) và Luật 6 (Ưu tiên Cao)
    # Mật độ cao(80), Tải thấp(20), Giao thông thấp(20) -> Combine Many
    # Khẩn cấp cao(80), Lợi nhuận cao(80) -> Priority High
    c, p = optimize_delivery(80, 80, 20, 20, 80)
    
    print(f"Kết hợp đơn: {c} (60-100 = Nhiều)")
    print(f"Ưu tiên giao: {p} (60-100 = Cao)")

=== 2.14 DELIVERY OPTIMIZATION ===
Kết hợp đơn: 83.11111111111109 (60-100 = Nhiều)
Ưu tiên giao: 84.44444444444444 (60-100 = Cao)
